# HackerNews Analytics Pro v5.0 — Live Monitoring

Fetch public Hacker News feeds, persist repeated score/comment observations in SQLite, and identify growth between polls. API access requires internet. Use sensible polling intervals.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT).exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from hn_analytics import HackerNewsClient, clean_stories, save_snapshot, growth_alerts


## 1. Fetch live top stories
The request is bounded to 30 items for a lightweight demonstration.


In [ ]:
client=HackerNewsClient()
raw=client.fetch_feed('top', limit=30, pause=0.1)
df=clean_stories(raw)
df[['id','title','score','descendants','domain']].head(10)


## 2. Persist a monitoring snapshot
Run this cell again later to compare the same stories. The SQLite database stores observations over time.


In [ ]:
db=ROOT/'hn_monitor.sqlite'
saved=save_snapshot(db, raw)
print(f'Stored {saved} observations in {db}')


## 3. Detect story growth
Alerts are threshold-based changes between earliest and latest stored observations in the lookback window, not predictions.


In [ ]:
alerts=growth_alerts(db, min_score_gain=20, min_comment_gain=10, lookback_hours=24)
display(alerts.sort_values('score_gain',ascending=False))
